# Практический Workbook: Exploratory Data Analysis

Этот notebook сопровождает автономный HTML-курс EDA. Он построен как исследование синтетических снимков клиентской базы.

**Файлы рядом с notebook:** `eda_practice_dataset.csv` и этот `.ipynb`.

**Единица наблюдения:** один клиент в один момент ежемесячного снимка (`customer_id`, `snapshot_date`). Один клиент может встречаться несколько раз. `churn_30d` — факт оттока в следующие 30 дней; в последнем месяце метка намеренно не созрела и записана как пропуск.

Датасет содержит специально добавленные дефекты: пропуски, выбросы, некорректные категории и повторные ключи. Он синтетический и не описывает реальных клиентов.

## 0. Настройка окружения

Для запуска установи зависимости из `requirements-eda.txt`. Случайные процессы зафиксированы seed, чтобы основные результаты воспроизводились.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
sns.set_theme()

DATA_PATH = Path("eda_practice_dataset.csv")
assert DATA_PATH.exists(), f"Не найден файл: {DATA_PATH.resolve()}"
df_raw = pd.read_csv(DATA_PATH, parse_dates=["snapshot_date"])
print("shape:", df_raw.shape)
df_raw.head()

## 1. Паспорт данных и единица анализа

Сначала ответь на вопрос: что представляет строка? Ключ наблюдения — пара «клиент × дата снимка». `customer_id` не является уникальным ключом всей таблицы.

Проверь диапазон дат, число клиентов и число снимков на клиента. Не удаляй повторные строки до выяснения их происхождения.

In [ ]:
print(df_raw.dtypes)
print("period:", df_raw["snapshot_date"].min(), "—", df_raw["snapshot_date"].max())
print("customers:", df_raw["customer_id"].nunique())
print("snapshots per customer:")
display(df_raw.groupby("customer_id").size().describe())
KEY = ["customer_id", "snapshot_date"]
print("duplicate business keys:", df_raw.duplicated(KEY, keep=False).sum())

**Задание 1.** Объясни, почему `customer_id` нельзя использовать для `drop_duplicates`, и чем отличается точный дубликат строки от конфликта ключа, когда значения в повторившихся строках различаются.

In [ ]:
# Сначала исследуем группы конфликтующих ключей.
conflicts = (df_raw[df_raw.duplicated(KEY, keep=False)]
             .sort_values(KEY))
display(conflicts.head(20))
print("exact duplicate rows:", df_raw.duplicated().sum())

## 2. Создание аналитической копии и аудит качества

Оригинал сохраняем неизменным. Для примера удаляем повторные ключи по явно выбранному правилу «оставить первую запись». В реальном проекте это правило должно исходить из источника данных и временных меток обновления.

In [ ]:
df = df_raw.copy()
df = df.sort_values(KEY).drop_duplicates(KEY, keep="first").reset_index(drop=True)
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_n": df.isna().sum(),
    "missing_pct": (df.isna().mean()*100).round(2),
    "n_unique": df.nunique(dropna=True),
})
display(quality.sort_values("missing_pct", ascending=False))

**Задание 2.** Найди недопустимые значения: отрицательные сессии, отрицательные тарифы, отрицательные сроки активности. Сформулируй, какие значения точно ошибочны, а какие могут быть редкими, но реальными.

In [ ]:
checks = {
    "negative_sessions": df["sessions_30d"] < 0,
    "negative_charge": df["monthly_charge"] < 0,
    "negative_tenure": df["tenure_months"] < 0,
    "unknown_payment": ~df["payment_method"].isin(["card", "direct_debit", "wallet", "bank_transfer"]),
}
for name, mask in checks.items():
    print(f"{name}: {int(mask.sum())}")

## 3. Механизмы пропусков и зрелость метки

Пропуск target в последнем месяце — не случайная поломка измерения, а административное цензурирование: для этих наблюдений 30-дневное окно ещё не завершилось. Их нельзя механически считать классом `0`.

Сравни долю пропусков по времени и признакам. Данные не позволяют автоматически доказать MCAR, MAR или MNAR; эти механизмы требуют предположений о процессе сбора.

In [ ]:
df["snapshot_month"] = df["snapshot_date"].dt.to_period("M").astype(str)
print(df.groupby("snapshot_month")["churn_30d"].agg(rows="size", label_missing=lambda x: x.isna().sum(), observed_rate=lambda x: x.mean()))
missing_by_plan = df.groupby("plan_type")["monthly_charge"].apply(lambda x: x.isna().mean()).rename("missing_rate")
display(missing_by_plan)

**Задание 3.** Построй таблицу долей пропусков для всех столбцов по `snapshot_month`. Отдельно укажи структурные пропуски target и обычные пропуски признаков.

In [ ]:
missing_by_month = (df.groupby("snapshot_month").apply(lambda g: g.isna().mean(), include_groups=False)*100)
display(missing_by_month.round(1))

## 4. Одномерные распределения и робастные статистики

Сравни mean, median, стандартное отклонение и IQR. Выбросы могут сильно сдвигать среднее, но медиана и IQR также не универсальны: выбор характеристики зависит от задачи.

In [ ]:
numeric = ["tenure_months", "monthly_charge", "sessions_30d", "support_tickets_30d", "days_since_last_login"]
display(df[numeric].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).T)
summary = pd.DataFrame({
    "mean": df[numeric].mean(),
    "median": df[numeric].median(),
    "std": df[numeric].std(),
    "IQR": df[numeric].quantile(.75)-df[numeric].quantile(.25),
})
display(summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["monthly_charge"].dropna(), bins=40)
axes[1].boxplot(df["monthly_charge"].dropna(), vert=False)
axes[0].set_title("Monthly charge: histogram")
axes[1].set_title("Monthly charge: boxplot")
plt.tight_layout()

**Задание 4.** Выбери один признак с асимметрией или длинным хвостом. Объясни, почему среднее может быть плохим «типичным значением». Укажи, какие наблюдения нужно проверить по первоисточнику.

## 5. Категории, сегменты и условные распределения

In [ ]:
for col in ["plan_type", "region", "payment_method", "contract_months"]:
    print("\n", col)
    display(df[col].value_counts(dropna=False).head(15))

segment = (df.groupby("plan_type", dropna=False)
             .agg(rows=("customer_id", "size"),
                  customers=("customer_id", "nunique"),
                  median_charge=("monthly_charge", "median"),
                  mean_sessions=("sessions_30d", "mean")))
display(segment)

**Задание 5.** Сравни отток между тарифами, но не ограничивайся агрегированным процентом: посмотри на контракт, tenure и дату снимка. Напиши минимум одно альтернативное объяснение обнаруженной разницы.

In [ ]:
mature = df[df["churn_30d"].notna()].copy()
churn_by_plan = mature.groupby("plan_type")["churn_30d"].agg(["mean", "count"])
display(churn_by_plan)
(mature.groupby("plan_type")["churn_30d"].mean()
 .plot(kind="bar", title="Crude 30-day churn rate by plan"))
plt.ylabel("Observed rate — not a causal effect")
plt.tight_layout()
plt.show()

## 6. Зависимости и неопределённость

Корреляция описывает определённую форму связи, а не причинный эффект. Пирсон чувствителен к нелинейности и выбросам; Спирмен использует ранги и отвечает на вопрос о монотонной связи.

In [ ]:
corr = mature[numeric + ["churn_30d"]].corr(method="spearman")
sns.heatmap(corr, annot=True, fmt=".2f", center=0, cmap="vlag")
plt.title("Spearman rank correlations")
plt.show()

# Bootstrap CI для средней месячной платы — ресэмплируем строки как независимые только для демонстрации.
x = mature["monthly_charge"].dropna().to_numpy()
boot = np.array([np.mean(np.random.choice(x, size=len(x), replace=True)) for _ in range(2000)])
print("mean:", x.mean(), "95% percentile bootstrap CI:", np.quantile(boot, [.025,.975]))
print("Важно: повторные снимки одного клиента зависимы; для строгого CI нужен cluster bootstrap по customer_id.")

**Задание 6.** Повтори bootstrap, но ресэмплируй клиентов целыми кластерами, а не отдельные строки. Сравни ширину интервалов.

In [ ]:
# Cluster bootstrap: выбор клиентов с возвращением, затем все их строки.
client_ids = mature["customer_id"].unique()
# Готовим массивы по клиентам один раз; это быстрее, чем фильтровать DataFrame в каждом повторе.
client_values = [g["monthly_charge"].dropna().to_numpy() for _, g in mature.groupby("customer_id")]
client_values = [x for x in client_values if len(x)]
cluster_boot = []
for _ in range(500):
    sampled_idx = np.random.randint(0, len(client_values), size=len(client_values))
    sample_values = np.concatenate([client_values[i] for i in sampled_idx])
    cluster_boot.append(sample_values.mean())
print("cluster bootstrap CI:", np.quantile(cluster_boot, [.025,.975]))


## 7. Временная структура и созревание target

In [ ]:
monthly = (df.groupby("snapshot_month")
            .agg(rows=("customer_id", "size"),
                 customers=("customer_id", "nunique"),
                 observed_target=("churn_30d", "count"),
                 churn_rate=("churn_30d", "mean")))
display(monthly)
monthly["churn_rate"].plot(marker="o", figsize=(9,4), title="30-day churn by snapshot month")
plt.ylabel("Rate among mature labels")
plt.show()

**Задание 7.** Почему сравнивать месяц с неполной зрелостью меток нельзя без дополнительной обработки? Что произойдёт, если записать все неизвестные метки как ноль?

## 8. Target construction, leakage и разбиение

До моделирования необходимо проверить, что все признаки доступны на `snapshot_date`, а target относится к будущему окну. В этом датасете `customer_id` — группирующий идентификатор, а `snapshot_date` — временная ось. Случайное разбиение строк может поместить снимки одного клиента одновременно в train и validation.

In [ ]:
mature = df[df["churn_30d"].notna()].copy()
FEATURES = ["plan_type", "region", "payment_method", "tenure_months", "monthly_charge",
            "sessions_30d", "support_tickets_30d", "days_since_last_login",
            "discount_active", "contract_months"]
TARGET = "churn_30d"
# Строгий temporal holdout: май — тестовый период, январь–апрель — обучение.
train = mature[mature["snapshot_date"] < "2025-05-01"].copy()
test = mature[mature["snapshot_date"] >= "2025-05-01"].copy()
print("train/test:", train.shape, test.shape)
print("date ranges:", train.snapshot_date.min(), train.snapshot_date.max(), test.snapshot_date.min(), test.snapshot_date.max())
print("customer overlap:", len(set(train.customer_id) & set(test.customer_id)))

Если продукт прогнозирует будущее для уже известных клиентов, пересечение customer_id может быть допустимым при строго временном тесте. Если задача — обобщать на новых клиентов, нужен Group split. Validation design должен моделировать реальный сценарий эксплуатации.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

cat_features = ["plan_type", "region", "payment_method"]
num_features = [c for c in FEATURES if c not in cat_features]
preprocess = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_features),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_features),
])
model = Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))])
model.fit(train[FEATURES], train[TARGET].astype(int))
proba = model.predict_proba(test[FEATURES])[:, 1]
y_test = test[TARGET].astype(int)
print("ROC-AUC:", roc_auc_score(y_test, proba))
print("Average precision:", average_precision_score(y_test, proba))

**Задание 8.** Сравни ROC-AUC и Average Precision. При редком положительном классе объясни, почему Average Precision может быть более информативной. Построй метрики по тарифам и месяцам, не подбирая порог на test.

In [ ]:
test_results = test[["customer_id", "snapshot_date", "plan_type", TARGET]].copy()
test_results["score"] = proba
test_results["prediction_050"] = (proba >= .5).astype(int)
segment_metrics = test_results.groupby("plan_type").apply(
    lambda g: pd.Series({"n": len(g), "positive_rate": g[TARGET].mean(),
                         "mean_score": g.score.mean(), "recall_at_050": ((g.prediction_050==1)&(g[TARGET]==1)).sum()/max(1,(g[TARGET]==1).sum())}),
    include_groups=False)
display(segment_metrics)

## 9. Исследовательский отчёт и следующие гипотезы

Заверши анализ не перечнем графиков, а коротким документом: контекст, единица наблюдения, качество и ограничения, 5 главных находок, 3 риска, validation design, baseline, error analysis, 3 следующих эксперимента.

In [ ]:
findings = [
    {"finding": "В данных есть повторные customer-month ключи", "evidence": "duplicate key audit", "risk": "повторное взвешивание наблюдений", "next_step": "установить правило дедупликации по журналу источника"},
    {"finding": "Целевая метка последнего месяца не созрела", "evidence": "missing target by snapshot_month", "risk": "смещение оценки churn rate", "next_step": "исключать незрелые окна или дождаться меток"},
]
findings_df = pd.DataFrame(findings)
display(findings_df)
findings_df.to_csv("eda_findings_template.csv", index=False)
print("Workbook завершён. Заполни findings своими результатами, а не оставляй демонстрационные формулировки.")

## Контрольные вопросы

1. Почему одна строка не равна одному клиенту?
2. Почему target пропуск в июне нельзя заполнить нулём?
3. Что отличает точный дубликат от конфликта бизнес-ключа?
4. Когда медиана предпочтительнее среднего?
5. Что измеряет Spearman correlation?
6. Почему bootstrap по строкам некорректен при кластерной зависимости?
7. В каких условиях random split создаёт утечку между клиентами?
8. Чем отличается covariate shift от concept drift?
9. Почему crude churn rate по тарифам не является причинным эффектом?
10. Какие доказательства должны сопровождать аналитический вывод?